In [1]:
import numpy as np
import pandas as pd
import tensorflow as tf

/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/numpy/_core/getlimits.py:551: UserWarning: Signature b'\x00\xd0\xcc\xcc\xcc\xcc\xcc\xcc\xfb\xbf\x00\x00\x00\x00\x00\x00' for <class 'numpy.longdouble'> does not match any known type: falling back to type probe function.
This warnings indicates broken support for the dtype!
  machar = _get_machar(dtype)
2026-07-18 11:11:34.274578: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [2]:
df = pd.read_csv('cardio_train1.csv')
df.head()

,id,age,gender,height,weight,ap_hi,ap_lo,cholesterol,gluc,smoke,alco,active,cardio
0,0,18393,2,168,62.0,110,80,1,1,0,0,1,0
1,1,20228,1,156,85.0,140,90,3,1,0,0,1,1
2,2,18857,1,165,64.0,130,70,3,1,0,0,0,1
3,3,17623,2,169,82.0,150,100,1,1,0,0,1,1
4,4,17474,1,156,56.0,100,60,1,1,0,0,0,0


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 70000 entries, 0 to 69999
Data columns (total 13 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   id           70000 non-null  int64  
 1   age          70000 non-null  int64  
 2   gender       70000 non-null  int64  
 3   height       70000 non-null  int64  
 4   weight       70000 non-null  float64
 5   ap_hi        70000 non-null  int64  
 6   ap_lo        70000 non-null  int64  
 7   cholesterol  70000 non-null  int64  
 8   gluc         70000 non-null  int64  
 9   smoke        70000 non-null  int64  
 10  alco         70000 non-null  int64  
 11  active       70000 non-null  int64  
 12  cardio       70000 non-null  int64  
dtypes: float64(1), int64(12)
memory usage: 6.9 MB


In [4]:
df.drop('id', axis=1, inplace=True)

In [5]:
X = df.drop('cardio', axis=1)
y = df['cardio']

In [6]:
from sklearn.preprocessing import StandardScaler

SS = StandardScaler()
X = SS.fit_transform(X)

In [7]:
X.shape

(70000, 11)

In [8]:
y.shape

(70000,)

In [9]:
y = pd.get_dummies(y)

In [31]:
y.head()

,0,1
0,True,False
1,False,True
2,False,True
3,False,True
4,True,False


In [10]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [11]:
model = tf.keras.models.Sequential([
    tf.keras.layers.Dense(11, activation='relu', input_shape=(11, )),
    tf.keras.layers.Dense(30, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])

/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
2026-07-18 11:11:42.505280: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


In [12]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 11)             │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 30)             │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 2)              │            62 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 554 (2.16 KB)

 Trainable params: 554 (2.16 KB)

 Non-trainable params: 0 (0.00 B)

In [13]:
from tensorflow.keras.utils import plot_model

In [14]:
plot_model(model)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.


In [16]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[tf.keras.metrics.CategoricalAccuracy()]
)

In [17]:
model.fit(X_train, y_train, batch_size=100, epochs=10, validation_data=(X_test, y_test))

Epoch 1/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 5s 4ms/step - categorical_accuracy: 0.6367 - loss: 0.6367 - val_categorical_accuracy: 0.6637 - val_loss: 0.6142
Epoch 2/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - categorical_accuracy: 0.6730 - loss: 0.6055 - val_categorical_accuracy: 0.6941 - val_loss: 0.5890
Epoch 3/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - categorical_accuracy: 0.7105 - loss: 0.5775 - val_categorical_accuracy: 0.7198 - val_loss: 0.5720
Epoch 4/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - categorical_accuracy: 0.7228 - loss: 0.5644 - val_categorical_accuracy: 0.7248 - val_loss: 0.5636
Epoch 5/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - categorical_accuracy: 0.7271 - loss: 0.5577 - val_categorical_accuracy: 0.7306 - val_loss: 0.5603
Epoch 6/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - categorical_accuracy: 0.7297 - loss: 0.5520 - val_categorical_accuracy: 0.7312 - val_loss: 0.5548
Epoch 7/10
560/560 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - categorical_accuracy: 0.7295 - los

In [18]:
from sklearn.ensemble import RandomForestClassifier

In [19]:
yr = df['cardio']

In [20]:
Xr_train, Xr_test, yr_train, yr_test = train_test_split(X, yr, test_size=0.2, random_state=42)

In [21]:
RF = RandomForestClassifier(n_estimators=300)
r_model = RF.fit(Xr_train, yr_train)

In [22]:
y_hat = r_model.predict(Xr_test)

In [25]:
from sklearn.metrics import accuracy_score, classification_report

In [24]:
accuracy_score(yr_test, y_hat)

0.7174285714285714

In [26]:
print(classification_report(yr_test, y_hat))

              precision    recall  f1-score   support

           0       0.71      0.73      0.72      6988
           1       0.72      0.71      0.71      7012

    accuracy                           0.72     14000
   macro avg       0.72      0.72      0.72     14000
weighted avg       0.72      0.72      0.72     14000



## Try with the deepest NN model

In [27]:
model = tf.keras.Sequential([
    tf.keras.layers.Dense(11, activation='relu', input_shape=(11,)),
    tf.keras.layers.Dense(30, activation='relu'),
    tf.keras.layers.Dense(60, activation='relu'),
    tf.keras.layers.Dense(25, activation='relu'),
    tf.keras.layers.Dense(2, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
    loss=tf.keras.losses.CategoricalCrossentropy(),
    metrics=[tf.keras.metrics.CategoricalAccuracy()]
)

/home/lucifer666/anaconda3/envs/py312/lib/python3.12/site-packages/keras/src/layers/core/dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
model.fit(X_train, y_train, batch_size=100, epochs=50, validation_data=(X_test, y_test))

Epoch 1/50
560/560 ━━━━━━━━━━━━━━━━━━━━ 4s 4ms/step - categorical_accuracy: 0.6745 - loss: 0.6083 - val_categorical_accuracy: 0.7185 - val_loss: 0.5769
Epoch 2/50
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - categorical_accuracy: 0.7190 - loss: 0.5669 - val_categorical_accuracy: 0.7287 - val_loss: 0.5592
Epoch 3/50
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - categorical_accuracy: 0.7245 - loss: 0.5580 - val_categorical_accuracy: 0.7336 - val_loss: 0.5590
Epoch 4/50
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - categorical_accuracy: 0.7280 - loss: 0.5538 - val_categorical_accuracy: 0.7312 - val_loss: 0.5517
Epoch 5/50
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - categorical_accuracy: 0.7283 - loss: 0.5500 - val_categorical_accuracy: 0.7321 - val_loss: 0.5488
Epoch 6/50
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - categorical_accuracy: 0.7293 - loss: 0.5481 - val_categorical_accuracy: 0.7321 - val_loss: 0.5502
Epoch 7/50
560/560 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - categorical_accuracy: 0.7292 - los

In [29]:
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 11)             │           132 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 30)             │           360 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 60)             │         1,860 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 25)             │         1,525 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 2)              │            52 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,789 (46.05 KB)

 Trainable params: 3,929 (15.35 KB)

 Non-trainable params: 0 (0.00 B)

 Optimizer params: 7,860 (30.71 KB)

In [30]:
model.predict(X_test)[3]

438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


array([0.67183733, 0.32816258], dtype=float32)

In [32]:
np.argmax(model.predict(X_test)[3])

438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


np.int64(0)

In [33]:
y_pred = np.argmax(model.predict(X_test), axis=1)

438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step


In [35]:
print(classification_report(yr_test, y_pred))

              precision    recall  f1-score   support

           0       0.72      0.79      0.75      6988
           1       0.77      0.69      0.73      7012

    accuracy                           0.74     14000
   macro avg       0.74      0.74      0.74     14000
weighted avg       0.74      0.74      0.74     14000

